# 07 · 풀 학습 결과 시각화 — 세 모델 정성/정량 비교

> **공부 기록 노트북 7번.** 06 은 *"파이프라인이 끝까지 돌아가는가"* 를 봤습니다.
> 7번은 그 위 단계 — **실제로 학습된 세 모델**(`unet_baseline` / `sam2_lora` /
> `sam2_temporal`) 의 결과를 같은 프레임에서 *나란히* 비교합니다.

## 이 노트북이 읽어 오는 것

학습이 끝나면 `src/train/train_segmentation.py` 가 모델별로 베스트 체크포인트를 씁니다:

```
outputs/unet_baseline/best.ckpt
outputs/sam2_lora/best.ckpt
outputs/sam2_temporal/best.ckpt
```

이 노트북은 **정확히 그 세 경로**를 읽습니다. 즉:

- **RunPod (학습한 그 머신) 에서 그대로 실행** → 추가 작업 0, 그냥 돌리면 됩니다
- **다른 머신 (예: Colab) 에서 시각화** → `outputs/` 폴더를 그 머신으로 가져오기만 하면 됩니다

> 학습을 아직 안 끝냈으면 이 노트북은 첫 체크 셀에서 친절한 에러로 멈춥니다.
> 06 (스모크) 만 돌리고 와도 베스트 ckpt 는 있으니, 결과가 *낮은 것 외엔* 동작합니다.

## 순서

1. 환경 + 체크포인트 존재 확인
2. 세 모델 로드 (config + ckpt)
3. **(핵심) 5 열 나란히 비교**: `[input | GT | U-Net | SAM2+LoRA | +temporal]`
4. 벤치마크 표의 per-class Dice 막대 비교
5. (선택) temporal 모델의 프레임 간 일관성 시각화

In [ ]:
%cd /content
import os
if not os.path.isdir("surgical-ai"):
    !git clone -b main https://github.com/duck-bin/surgical-ai.git
%cd surgical-ai
!git pull --ff-only
!bash scripts/colab_setup.sh

import torch
print("준비 완료 ·", os.getcwd())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (시각화만 할 거라 OK)")

## 1. 체크포인트 존재 확인

학습이 끝났다면 아래 세 파일이 모두 있어야 합니다. 없는 게 있으면 어떤 게 빠졌는지
알려주고 멈춥니다. (06 스모크만 돌리고 오셨어도 세 파일 다 만들어집니다 — 점수만
낮을 뿐 시각화는 정상 동작합니다.)

In [ ]:
from pathlib import Path

MODELS = [
    ("U-Net",              "unet_baseline"),
    ("SAM2+LoRA",          "sam2_lora"),
    ("SAM2+LoRA+temporal", "sam2_temporal"),
]

missing = []
for label, name in MODELS:
    ckpt = Path(f"outputs/{name}/best.ckpt")
    print(f"{label:20s} {'OK' if ckpt.is_file() else 'MISSING'}  {ckpt}")
    if not ckpt.is_file():
        missing.append((label, ckpt))

if missing:
    raise FileNotFoundError(
        "\n다음 체크포인트가 없습니다 — 학습을 먼저 돌리거나 outputs/ 폴더를 옮겨 오세요:\n"
        + "\n".join(f"  - {label}: {ckpt}" for label, ckpt in missing)
    )

## 2. 세 모델 로드

각 모델의 config 를 읽어 빈 모델을 만들고, 그 위에 학습된 가중치를 얹습니다.
`build_model(..., pretrained=False)` 로 큰 SAM2 가중치 다운로드를 건너뛰고
*우리* 체크포인트를 strict 로드 — 벤치마크 코드가 쓰는 그 함수입니다.

In [ ]:
import torch
from omegaconf import OmegaConf

from src.train.train_segmentation import build_model
from src.eval.benchmark_runner import load_segmentation_checkpoint

device = "cuda" if torch.cuda.is_available() else "cpu"
loaded = []
for label, name in MODELS:
    cfg = OmegaConf.load(f"configs/model/{name}.yaml")
    model = build_model(cfg, pretrained=False)
    model = load_segmentation_checkpoint(model, f"outputs/{name}/best.ckpt")
    model = model.to(device).eval()
    loaded.append((label, name, cfg, model))
    print(f"loaded {label}  (config={name})")

## 3. (핵심) 한 프레임에 세 모델 나란히 — `[input | GT | U-Net | SAM2 | temporal]`

같은 val 프레임을 모든 모델에 똑같이 먹여, 다섯 열로 비교합니다. 색깔은
`src/utils/viz.py::CLASS_COLORS` 의 6 클래스 컬러팔레트:

| 색 | 클래스 |
|---|---|
| 빨강 | liver |
| 파랑 | gallbladder |
| 노랑 | **cystic_duct** ← 우리가 가장 관심 있는 클래스 |
| 초록 | cystic_artery (CholecSeg8k 엔 라벨 없음 → 항상 0) |
| 회색 | tool |

> 적은 epoch 으로 학습했으면 (06 스모크처럼) 예측이 거의 background 일 수 있습니다.
> *비교*가 목적이라 그 차이 자체가 정보입니다.

`SAMPLE_INDICES` 를 바꿔서 다른 프레임을 보실 수도 있습니다.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from src.data.cholecseg8k import CholecSeg8kDataset, CholecSeg8kWindowDataset
from src.data.transforms import build_eval_transforms
from src.utils.viz import overlay_mask

IMG = 256                        # 256 정도면 그림이 또렷하고 빠릅니다 (학습 해상도 무관)
SAMPLE_INDICES = [0, 50, 120]    # val 에서 그릴 프레임 (원하시면 바꿔보세요)

eval_tf = build_eval_transforms(IMG)
# Frame 기반 val 데이터셋 (U-Net / SAM2+LoRA 가 받는 형식)
frame_ds = CholecSeg8kDataset(split="val", image_size=IMG, transform=eval_tf)

# temporal 모델은 윈도우를 받습니다. 같은 타깃 프레임을 보장하기 위해 frame_ds 의
# 인덱스 대신, temporal 윈도우들 중 *타깃이 같은* 것을 매칭해 사용합니다.
window = int(loaded[2][2].get("window", 3))
window_ds = CholecSeg8kWindowDataset(split="val", window=window,
                                     image_size=IMG, transform=eval_tf)

def _denorm(img_t):
    """학습 정규화를 되돌려 시각화용 uint8 RGB 로."""
    mean = np.array([0.485, 0.456, 0.406]); std = np.array([0.229, 0.224, 0.225])
    arr = img_t.permute(1, 2, 0).numpy() * std + mean
    return (np.clip(arr, 0, 1) * 255).astype(np.uint8)

@torch.no_grad()
def _predict(label, model, sample):
    x = sample["image"].unsqueeze(0).to(device)
    return model(x).argmax(dim=1)[0].cpu().numpy()

fig, ax = plt.subplots(len(SAMPLE_INDICES), 5,
                       figsize=(20, 4 * len(SAMPLE_INDICES)))
if len(SAMPLE_INDICES) == 1:
    ax = ax[np.newaxis, :]
col_titles = ["input", "ground truth"] + [m[0] for m in MODELS]

for row, idx in enumerate(SAMPLE_INDICES):
    frame_sample = frame_ds[idx]
    rgb = _denorm(frame_sample["image"])
    gt = frame_sample["mask"].numpy()

    ax[row, 0].imshow(rgb)
    ax[row, 1].imshow(overlay_mask(rgb, gt))

    # U-Net / SAM2+LoRA: frame 입력
    for col, (label, name, cfg, model) in enumerate(loaded[:2], start=2):
        pred = _predict(label, model, frame_sample)
        ax[row, col].imshow(overlay_mask(rgb, pred))

    # temporal: 같은 타깃 프레임에 해당하는 윈도우 샘플 사용
    # (윈도우 데이터셋은 frame_ds 의 *local* 인덱스를 마지막 프레임으로 갖는 윈도우만 유효)
    matched = None
    for w_idx in range(len(window_ds)):
        if window_ds._windows[w_idx][-1] == idx:
            matched = window_ds[w_idx]; break
    if matched is not None:
        pred = _predict(loaded[2][0], loaded[2][3], matched)
        ax[row, 4].imshow(overlay_mask(rgb, pred))
    else:
        ax[row, 4].text(0.5, 0.5,
                        f"no window\n(frame {idx} too\nearly in video)",
                        ha="center", va="center", fontsize=10,
                        transform=ax[row, 4].transAxes)

    if row == 0:
        for col, title in enumerate(col_titles):
            ax[row, col].set_title(title)
    for col in range(5):
        ax[row, col].axis("off")

plt.tight_layout(); plt.show()

## 4. 벤치마크 표의 per-class Dice 막대 비교

`benchmark_runner` 가 이미 써준 `results/benchmark_table.md` 에 같은 정보가 있지만,
표만 보면 한눈에 안 들어옵니다. 같은 데이터(이번 세션 또는 디스크에 있는)를
**모델별 per-class Dice 막대그래프**로 그립니다 — `cystic_duct` 막대를 특히
보세요.

In [ ]:
from pathlib import Path
import json
import re

# 벤치마크 표(텍스트)에서 per-class Dice 를 파싱해 그립니다.
# 표가 없으면 셀이 친절히 안내하고 스킵.
path = Path("results/benchmark_table.md")
if not path.is_file():
    print("results/benchmark_table.md 가 아직 없습니다 -- 먼저 다음을 돌리세요:")
    print("  python -m src.eval.benchmark_runner data.image_size=256 batch_size=1")
else:
    text = path.read_text()
    # 'Per-class Dice' 섹션 추출
    section = text.split("Per-class Dice")[-1]
    rows = [r.strip() for r in section.splitlines() if r.strip().startswith("|")]
    rows = [r for r in rows if not set(r.replace("|", "").strip()) <= set("-: ")]
    header = [h.strip() for h in rows[0].strip("|").split("|")]
    model_labels = header[1:]
    cls, mat = [], []
    for r in rows[1:]:
        parts = [p.strip() for p in r.strip("|").split("|")]
        cls.append(parts[0])
        # "0.123 (0.110-0.135)" 또는 "TBD" -> float / NaN
        def _val(cell):
            m = re.match(r"([0-9.]+)", cell)
            return float(m.group(1)) if m else float("nan")
        mat.append([_val(c) for c in parts[1:]])
    import numpy as np
    mat = np.array(mat)

    x = np.arange(len(cls)); w = 0.8 / len(model_labels)
    fig, ax = plt.subplots(figsize=(11, 5))
    for i, label in enumerate(model_labels):
        ax.bar(x + (i - len(model_labels)/2 + 0.5) * w, mat[:, i], w, label=label)
    ax.set_xticks(x); ax.set_xticklabels(cls, rotation=15)
    ax.set_ylabel("Dice"); ax.set_ylim(0, 1.0)
    ax.set_title("Per-class Dice -- 세 모델 비교 (CholecSeg8k 테스트)")
    ax.legend(); ax.grid(axis="y", linestyle=":", alpha=0.4)
    plt.tight_layout(); plt.show()

## 5. (선택) Temporal 모델의 프레임 간 일관성

`SAM2+LoRA+temporal` 의 핵심 주장은 *frame-to-frame consistency 개선*입니다 —
얇은 구조(특히 `cystic_duct`) 가 인접 프레임에서 flicker 가 덜하게.

같은 비디오에서 연속한 프레임 N 개를 뽑아 두 줄로 비교합니다:

- 윗줄: `SAM2+LoRA` 예측 (프레임 독립)
- 아랫줄: `+temporal` 예측 (윈도우)

옆 프레임끼리 *비슷한지* 가 핵심 — 색이 마구 바뀌면 flicker, 잔잔하면 안정적입니다.

In [ ]:
from src.data.cholecseg8k import CholecSeg8kWindowDataset

CONSEC_N = 5
window = int(loaded[2][2].get("window", 3))
window_ds = CholecSeg8kWindowDataset(split="val", window=window,
                                     image_size=IMG, transform=eval_tf)

if len(window_ds) < CONSEC_N:
    print("윈도우가 부족합니다. val 에 충분한 연속 프레임이 없을 수 있어요.")
else:
    sam2_model    = loaded[1][3]
    temporal_model = loaded[2][3]
    fig, ax = plt.subplots(2, CONSEC_N, figsize=(4 * CONSEC_N, 8))

    for col in range(CONSEC_N):
        w_sample = window_ds[col]                                  # 연속 윈도우들
        target_frame = w_sample["image"][-1]                       # 마지막=타깃
        rgb = _denorm(target_frame)

        with torch.no_grad():
            sam2_pred = sam2_model(target_frame.unsqueeze(0).to(device)
                                   ).argmax(1)[0].cpu().numpy()
            tmp_pred = temporal_model(w_sample["image"].unsqueeze(0).to(device)
                                      ).argmax(1)[0].cpu().numpy()

        ax[0, col].imshow(overlay_mask(rgb, sam2_pred));  ax[0, col].axis("off")
        ax[1, col].imshow(overlay_mask(rgb, tmp_pred));   ax[1, col].axis("off")
        ax[0, col].set_title(f"frame {col}")
    ax[0, 0].set_ylabel("SAM2+LoRA",   fontsize=12)
    ax[1, 0].set_ylabel("+temporal",  fontsize=12)
    plt.suptitle("연속 프레임 — 프레임 간 일관성 비교 (색이 잔잔할수록 안정)")
    plt.tight_layout(); plt.show()

## 마무리

이 노트북이 *비어 있던 부분* — "그래서 어느 모델이 더 나은가" 를 **눈으로 확인**
시켜줍니다. 표(mIoU, cystic_duct Dice) 와 함께 보세요:

- **3 번 셀(나란히 비교)** → 모델별 마스크가 시각적으로 어떻게 다른지
- **4 번 셀(막대 그래프)** → 클래스별 강·약점이 수치로
- **5 번 셀(연속 프레임)** → temporal 의 안정성 주장이 실제로 보이는지

### 다음

- 결과가 만족스러우면 `train_cvs` → `app/gradio_demo.py` 로 인터랙티브 데모.
- 결과가 약하면 — 보통 *더 많은 epoch* 가 답입니다. cystic_duct 는 희소해서 100
  epoch 이상 가야 본격적으로 학습되기도 합니다.